In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import numpy as np
from scipy.integrate import solve_ivp
from sympy import symbols, sympify
import torch
import re

from halcyon.Models.MechanisticModel import ComplexODEModel
from halcyon.RegulatoryNetwork import RegulatoryNetwork
import libsbml
from sympy.parsing.sympy_parser import parse_expr
import sympy as sp

ModuleNotFoundError: No module named 'halcyon'

In [ ]:
# def test_import_from_sbml():
#     sbml_path = "tests/genenetweaver_outputs/test_scalefree.xml"
#     # my_importclass_test = SBMLImportClassDummy.from_sbml(sbml_path)
#     return my_importclass_test

In [ ]:
# imported = test_import_from_sbml()
# display(imported)

In [3]:
# folder_path = "tests/genenetweaver_outputs/"
folder_path = "tests/6_gene_testv2/"
sbml_path = folder_path + "test_scalefree.xml"
reader = libsbml.SBMLReader()
document = reader.readSBML(sbml_path)  # type: ignore
model = document.getModel()
print(model.getNumReactions())
parameter_list = model.getListOfParameters()
# To get the parameters of the model, you can use getListOfParameters()
parameters = model.getListOfParameters()
print(f"Number of parameters in the model: {parameters.size()}")
for i in range(parameters.size()):
    param = parameters.get(i)
    print(f"Parameter id: {param.getId()}, value: {param.getValue()}")

NameError: name 'libsbml' is not defined

In [ ]:
# Iterate through all reactions in the model and print their details
for i in range(model.getNumReactions()):
    reaction = model.getReaction(i)
    print(f"Reaction {i}: {reaction.getId()}")
    print(f"  Reactants: {[reactant.getSpecies() for reactant in reaction.getListOfReactants()]}")
    print(f"  Products: {[product.getSpecies() for product in reaction.getListOfProducts()]}")
    print(f"  Kinetic Law: {reaction.getKineticLaw().getFormula() if reaction.getKineticLaw() else 'None'}")
    print()

In [2]:
alternative_max_name = "maxRate"


def from_sbml(sbml_path: str):
    # Import SBML file
    reader = libsbml.SBMLReader()
    document = reader.readSBML(sbml_path)  # type: ignore
    model = document.getModel()

    # prepare parameter and function dictionaries
    param_dict = dict()
    ODE_dict = dict()

    #loop through reactions and parse them
    for i in range(model.getNumReactions()):
        reaction = model.getReaction(i)
        reaction_id = reaction.getId()#.rstrip("_")

        expression, params = __parse_one_reaction(reaction)
        # print(f"Parsed reaction {reaction_id}: {expression} with parameters {params}")
        ODE_dict[reaction_id] = expression
        
        # get separate parameters for each reaction, to be flattened later
        param_dict[reaction_id] = params
        
    # Flatten the parameter dictionary to match the expected format
    flat_param_dict = {}
    for node_name, params in param_dict.items():
        for pname, pval in params.items():
            flat_param_dict[pname] = pval

    # Get all node names from ODE_dict keys
    base_nodes = [node.replace("_degradation", "").replace("_synthesis", "") for node in ODE_dict.keys()] # get base node names without degradation/synthesis suffixes
    base_nodes = list(dict.fromkeys(base_nodes))  # remove duplicates
        
    # Combine synthesis and degradation equations for each node
    combined_ODE_dict = {}
    for node in base_nodes:
        synth_key = f"{node}_synthesis"
        degr_key = f"{node}_degradation"
        synth_expr = ODE_dict.get(synth_key, 0)
        degr_expr = ODE_dict.get(degr_key, 0)
        combined_ODE_dict[node] = synth_expr - degr_expr
        
    return combined_ODE_dict, flat_param_dict


def __parse_one_reaction(reaction: libsbml.Reaction):
    """_summary_

    Args:
        reaction (libsbml.Reaction): _description_

    Returns:
        Tuple of [Sympy expression, all_parameters]
    """
    kinetic_law = reaction.getKineticLaw()
    params = kinetic_law.getListOfParameters()
    param_dict = {}
    base_node_name = reaction.getId().replace("_degradation", "").replace("_synthesis", "")
    formula = kinetic_law.getFormula()
    
    # print(f"Processing reaction {reaction.getId()} with formula: {formula}")
    
    # Process parameters and update formula such that parameter names have the node name as suffix
    for param in params:
        # Skip unwanted parameters
        if param.getId().startswith("binds") or param.getId().startswith("num"):
            continue
            
        # Get original parameter name
        orig_param_id = param.getId()
        
        # Handle special case for 'max'
        if orig_param_id == "max":
            orig_param_id = alternative_max_name
            # Replace 'max' with alternative_max_name in formula
            formula = re.sub(r'\bmax\b', alternative_max_name, formula)
        
        # Create the final parameter ID with node name
        final_param_id = orig_param_id
        if not final_param_id.endswith(base_node_name):
            final_param_id = f"{orig_param_id}_{base_node_name}"
        
        # Store parameter in dictionary
        param_dict[final_param_id] = param.getValue()
        
        # Update formula to use the final parameter ID
        # We need to be careful to replace whole words only
        formula = re.sub(r'\b' + re.escape(orig_param_id) + r'\b', final_param_id, formula)
        
    
    # Parse the updated formula
    expression = parse_expr(formula)
    
    return expression, param_dict

NameError: name 'libsbml' is not defined

In [ ]:
ODE_dict, param_dict = from_sbml(folder_path + "test_scalefree.xml")
display(ODE_dict, param_dict)
display(type(ODE_dict["Node_1"]), ODE_dict["Node_1"])

In [ ]:
for node, eq in ODE_dict.items():
    display(f"{node}: {eq}")
    display(ODE_dict[node])

## convert to lamdas

In [ ]:
# # Assume state vector s is ordered as [Node_0, Node_1, ..., Node_9, cell]
# # Map node names to indices in s for substitution
# node_names = [f"Node_{i}" for i in range(10)]
# node_indices = {name: idx for idx, name in enumerate(node_names)}

# # Get symbolic ODEs and parameters from SBML
# symbolic_ODE_dict, sbml_param_dict = from_sbml(folder_path + "test_scalefree.xml")

# ODE_lambda_dict = {}

# for name, expr in symbolic_ODE_dict.items():
#     # Prepare the list of variables for substitution
#     def make_lambda(expr, param_dict, node_name):
#         symbols_in_expr = sorted(expr.free_symbols, key=lambda x: str(x))
#         # print(f"Symbols in expression for {node_name}: {symbols_in_expr}")
#         # Build a mapping from symbol name to param_dict key for this node
#         def func(s, param_dict):
#             subs = {}
#             for sym in symbols_in_expr:
#                 sym_name = str(sym)
#                 if sym_name in node_indices:
#                     subs[sym] = s[node_indices[sym_name]]
#                 elif sym_name == "cell":
#                     subs[sym] = s[-1]  # assuming cell is last in state vector
#                 else:
#                     # Try to find the correct param_dict key for this node
#                     direct_key = f"{sym_name}"
#                     if direct_key in param_dict:
#                         subs[sym] = param_dict[direct_key]
#                     else:
#                         raise KeyError(f"Symbol {sym_name} not found in state vector or parameters for node {node_name}.")
#             return float(expr.evalf(subs=subs))
#         return func

#     ODE_lambda_dict[name] = make_lambda(expr, sbml_param_dict, name)
    
# ODE_lambda_dict["Node_0"](np.zeros(11), sbml_param_dict)

### chatgpt's solution

In [ ]:
# Get symbolic ODEs and parameters from SBML
symbolic_ODE_dict, sbml_param_dict = from_sbml(folder_path + "test_scalefree.xml")
# Get node names from symbolic ODE dict keys
node_names = [name.replace("_degradation", "").replace("_synthesis", "") for name in symbolic_ODE_dict.keys()]
node_indices = {name: idx for idx, name in enumerate(node_names)}

In [ ]:
# def build_lambda_dict(symbolic_ODE_dict, node_indices):
#     ODE_lambda_dict = {}

#     for name, expr in symbolic_ODE_dict.items():
#         # collect all symbols in the expression
#         symbols_in_expr = sorted(expr.free_symbols, key=lambda x: str(x))

#         # make a lambdified function
#         func = sp.lambdify(symbols_in_expr, expr, modules="numpy")

#         # wrapper: maps (s, param_dict) -> float
#         def wrapped_func(s, param_dict, func=func, inputs=symbols_in_expr):
#             values = []
#             for sym in inputs:
#                 sym_name = str(sym)
#                 if sym_name in node_indices:
#                     values.append(s[node_indices[sym_name]])
#                 elif sym_name == "cell":
#                     values.append(s[-1])  # last index = "cell"
#                 else:  # parameter
#                     values.append(param_dict[sym_name])
#             out = func(*values)
#             return np.real_if_close(out)  # strip tiny complex parts
#         ODE_lambda_dict[name] = wrapped_func

#     return ODE_lambda_dict

# # # Get symbolic ODEs and parameters from SBML
# # symbolic_ODE_dict, sbml_param_dict = from_sbml(folder_path + "test_scalefree.xml")
# # # Get node names from symbolic ODE dict keys
# # node_names = [name.replace("_degradation", "").replace("_synthesis", "") for name in symbolic_ODE_dict.keys()]
# # node_indices = {name: idx for idx, name in enumerate(node_names)}

# lambda_dict = build_lambda_dict(symbolic_ODE_dict, node_indices)
# # Test the lambda function for Node_0
# s = np.ones(len(node_names) + 1)  # +1 for the "cell" variable
# # s[-1] = 1.0  # set cell variable to 1.0
# result = lambda_dict["Node_2"](s, sbml_param_dict)
# print(f"Result for Node_2: {result}")

In [ ]:
# display()
# # Test with a zero vector
# ODE_lambda_dict["Node_2"](np.ones(11), param_dict)

In [ ]:
# display(type(ODE_dict["Node_0"]))
# symbols_in_expr = ODE_dict["Node_0"].free_symbols
# display(symbols_in_expr)
# subs = {}
# for sym in symbols_in_expr:
# 	sym_name = str(sym)
# 	if sym_name in node_indices:
# 		subs[sym] = 0.0  # zero vector for nodes
# 	elif sym_name == "cell":
# 		subs[sym] = 0.0  # zero for cell
# 	elif sym_name in param_dict:
# 		subs[sym] = param_dict[sym_name]
# 	else:
# 		# fallback: try param_dict with node suffix
# 		found = False
# 		for k in param_dict:
# 			if k.startswith(sym_name):
# 				subs[sym] = param_dict[k]
# 				found = True
# 				break
# 		if not found:
# 			subs[sym] = 0.0  # default to zero if not found
# result = ODE_dict["Node_0"].evalf(subs=subs)
# display(result)

In [ ]:
# display(ODE_lambda_dict["Node_0"])

In [ ]:
# ODE_dict

In [ ]:
# replace each value in param dict with zero for testing
# display(ODE_lambda_dict["Node_0"](np.ones(11), param_dict), param_dict) 
# param_dict_KO = {k: 0.0 if re.match(r'^a_\d+_', k) else v for k, v in param_dict.items()}
# display(ODE_lambda_dict["Node_0"](np.ones(11), param_dict_KO), param_dict_KO)  

In [ ]:
param_dict.keys()

### chatgpt's vectorised solution, no dictionairy anymore but just one complete ODE function

In [ ]:
# def build_vectorized_system(symbolic_ODE_dict, param_dict):
#     # Generate node names dynamically from the keys of symbolic_ODE_dict
#     node_names = list(symbolic_ODE_dict.keys())

#     # Generate node_indices dynamically
#     node_indices = {name: idx for idx, name in enumerate(node_names)}
    
#     # Collect all equations in consistent order
#     exprs = [symbolic_ODE_dict[name] for name in node_names]

#     # Gather all symbols appearing anywhere
#     all_symbols = set().union(*(expr.free_symbols for expr in exprs))
#     all_symbols = sorted(all_symbols, key=lambda x: str(x))

#     # Lambdify once
#     func = sp.lambdify(all_symbols, exprs, modules="numpy")

#     # # Build wrapper
#     # def ode_system(t, s):
#     #     values = []
#     #     for sym in all_symbols:
#     #         sym_name = str(sym)
#     #         if sym_name in node_indices:
#     #             values.append(s[node_indices[sym_name]])
#     #         elif sym_name == "cell":
#     #             values.append(s[-1])
#     #         else:
#     #             values.append(param_dict[sym_name])
#     #     out = func(*values)  # returns vector of dydt
#     #     out = np.real_if_close(np.array(out, dtype=float))
#     #     dydt = np.zeros(len(node_names) + 1)
#     #     dydt[:-1] = out
#     #     dydt[-1] = 0.0  # cell dynamics if any
#     #     return dydt
#     def ode_system(t, s):
#         values = []
#         for sym in all_symbols:
#             sym_name = str(sym)
#             if sym_name in node_indices:
#                 values.append(s[node_indices[sym_name]])
#             elif sym_name == "cell":
#                 print("Skipping cell dynamics")
#                 continue  # skip cell completely
#             else:
#                 values.append(param_dict[sym_name])
#         out = func(*values)
#         return np.real_if_close(np.array(out, dtype=float))

#     return ode_system
# ode_system = build_vectorized_system(symbolic_ODE_dict, sbml_param_dict)

In [ ]:
def build_vectorized_system(symbolic_ODE_dict, param_dict):
    # Generate node names dynamically from ODE dict keys
    node_names = list(symbolic_ODE_dict.keys())
    node_indices = {name: idx for idx, name in enumerate(node_names)}

    # Collect ODE expressions in the same order
    exprs = [symbolic_ODE_dict[name] for name in node_names]

    # Collect all unique symbols across all equations
    all_symbols = sorted(
        set().union(*(expr.free_symbols for expr in exprs)),
        key=lambda x: str(x)
    )

    # Lambdify once
    func = sp.lambdify(all_symbols, exprs, modules="numpy")

    def ode_system(t, s):
        print(f"s: {s}, t: {t}")
        # s = np.maximum(s, 0.0) # Ensure non-negative state variables
        values = []
        for sym in all_symbols:
            sym_name = str(sym)
            if sym_name in node_indices:
                values.append(s[node_indices[sym_name]])   # state variable
            elif sym_name == "cell":
                # Either treat as constant 1.0, or pull from param_dict
                # print(param_dict.get("cell", 1.0))
                values.append(param_dict.get("cell", 1.0))
                # continue
            else:
                # Parameter lookup
                if sym_name not in param_dict:
                    raise KeyError(f"Missing parameter {sym_name} in param_dict")
                values.append(param_dict[sym_name])

        out = func(*values)
        return np.real_if_close(np.array(out, dtype=float))
        # return out

    return ode_system
ode_system = build_vectorized_system(symbolic_ODE_dict, sbml_param_dict)

In [ ]:
sbml_param_dict

In [ ]:
ode_system(0, np.ones(10))

## use the ODE system

In [ ]:
# Use scipy's solve_ivp to integrate the ODE system using ODE_lambda_dict and sbml_param_dict

KONode = ""#"Node_9"
param_dict_KO = sbml_param_dict.copy()
if KONode != "" and KONode is not None:
    print(f"Knocking out {KONode} by setting its parameters to zero.")
    for key in param_dict_KO.keys():
        if key.startswith("a_") and key.endswith(KONode):
            param_dict_KO[key] = 0.0
else:
    print("No knockout node specified, no parameters changed.")

ode_system = build_vectorized_system(symbolic_ODE_dict, param_dict_KO)

# def ode_system(t, y):
#     # y: state vector [Node_0, Node_1, ..., Node_9, cell]
#     dydt = np.zeros(len(node_names))
#     s = np.append(y, 1.0)  # assume cell=1.0 for all time
#     for i, name in enumerate(node_names):
#         dydt[i] = ODE_lambda_dict[name](s, param_dict_KO)
#     return dydt

# y0 = np.zeros(len(node_names))  # initial conditions for nodes
# Load initial values from TSV file
init_df = pd.read_csv(folder_path + "test_scalefree_nonoise_wildtype.tsv", sep="\t")
# Assume columns are node_names and values are in the first row
y0 = init_df.loc[0, node_names].values.astype(float)
# y0 = np.zeros(len(node_names))  # initial conditions for nodes
print(f"Initial conditions: {y0}, length of y0: {len(y0)}")
t_span = (0, 100)
t_eval = np.linspace(t_span[0], t_span[1], 100001)

solution = solve_ivp(ode_system, t_span, y0, t_eval=t_eval, method="LSODA")
plt.figure(figsize=(10,6))
for i, name in enumerate(node_names):
    plt.plot(solution.t, solution.y[i], label=name)
plt.xlabel("Time")
plt.ylabel("Concentration")
plt.legend()
plt.title("ODE Simulation from SBML Model")
plt.show()

In [ ]:
t = 12.952151334488287
# s = [0.279644, 0.13783969, 0.25006143, 0.39311061, 0.04041368, 0.1960632]

s = np.zeros(len(node_names))
ode_system(t, s)

In [ ]:
# def integrateODE(mutant = "", duration = 1000):
#     """Integrate ODEs using the ODE_lambda_dict and sbml_param_dict"""
#     param_dict_KO = sbml_param_dict.copy()
#     # Set parameters to zero for the specified mutant node
#     for key in param_dict_KO.keys():
#         if key.startswith("a_") and key.endswith(mutant):
#             param_dict_KO[key] = 0.0
#     # print(f"Integrating ODEs with parameters: {param_dict_KO}")
    
#     # Define the ODE system
#     def ode_system(t, y):
#         dydt = np.zeros(len(node_names))
#         s = np.append(y, 1.0)  # assume cell=1.0 for all time
#         for i, name in enumerate(node_names):
#             dydt[i] = ODE_lambda_dict[name](s, param_dict_KO)
#         if np.any(np.iscomplex(dydt)):
#             print(f"Complex values at t={t}: {dydt}")
#         if np.any(~np.isfinite(dydt)):
#             print(f"Non-finite values at t={t}: {dydt}")
#         return np.real(dydt)
#         # return dydt

#     # y0 = np.zeros(len(node_names))  # initial conditions for nodes
#     init_df = pd.read_csv(folder_path + "test_scalefree_nonoise_wildtype.tsv", sep="\t")
#     # Assume columns are node_names and values are in the first row
#     y0 = init_df.loc[0, node_names].values.astype(float)
#     t_span = (0, duration)
#     t_eval = np.linspace(t_span[0], t_span[1], 200)

#     solution = solve_ivp(ode_system, t_span, y0, t_eval=t_eval, method="BDF")
    
    
#     return solution

In [ ]:
fig, axes = plt.subplots(len(node_names), 1, figsize=(10, 4 * len(node_names)), sharex=True)

y0_mutant = np.zeros(len(node_names))  # initial conditions
# init_df = pd.read_csv(folder_path + "test_scalefree_nonoise_wildtype.tsv", sep="\t")
# # Assume columns are node_names and values are in the first row
# y0_mutant = init_df.loc[0, node_names].values.astype(float)


for idx, mutant in enumerate(node_names):
    # Set parameters to zero for the specified mutant node
    param_dict_mutant = sbml_param_dict.copy()
    for key in param_dict_mutant.keys():
        if key.startswith("a_") and key.endswith(mutant):
            param_dict_mutant[key] = 0.0

    # # Define the ODE system for the mutant
    # def ode_system_mutant(t, y):
    #     dydt = np.zeros(len(node_names))
    #     s = np.append(y, 1.0)  # assume cell=1.0 for all time
    #     for i, name in enumerate(node_names):
    #         dydt[i] = ODE_lambda_dict[name](s, param_dict_mutant)
    #     return dydt

    # # Simulate the system
    # y0_mutant = y0.copy()  # initial conditions
    # t_span_mutant = (0, 100)
    # t_eval_mutant = np.linspace(t_span_mutant[0], t_span_mutant[1], 200)
    # solution_mutant = solve_ivp(ode_system_mutant, t_span_mutant, y0_mutant, t_eval=t_eval_mutant, method="LSODA")
    
    # Integrate ODEs for each mutant
    print(f"Integrating ODEs for mutant: {mutant}")
    # solution_mutant = integrateODE(mutant)
    solution_mutant = solve_ivp(build_vectorized_system(symbolic_ODE_dict, param_dict_mutant), (0, 100), y0_mutant, t_eval=np.linspace(0, 100, 201), method="RK45")

    # Plot the results
    for i, node in enumerate(node_names):
        axes[idx].plot(solution_mutant.t, solution_mutant.y[i], label=node)
    axes[idx].set_title(f"Mutant: {mutant}")
    axes[idx].set_ylabel("Concentration")
    axes[idx].legend(loc='upper right')

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
from halcyon.Datasets.ExpressionDataWithConditions import ExpressionDataWithConditions

input_path = Path(
        folder_path + "test_scalefree_nonoise_knockout_timeseries.tsv"
    )
KOData = ExpressionDataWithConditions.from_genenetweaver_ko_timeseries_output(
        input_path
    )

In [ ]:
# def plot_fit_vs_real(self):
# """Plot fitted model vs measured data"""
# assert len(dataset) > 0, "Dataset is empty. Cannot plot fit vs real."
from halcyon.Datasets.ExpressionData import ExpressionData
dataset = KOData

# y0 = np.zeros(len(node_names))  # initial conditions for nodes
init_df = pd.read_csv(folder_path + "test_scalefree_nonoise_wildtype.tsv", sep="\t")
# Assume columns are node_names and values are in the first row
y0 = init_df.loc[0, node_names].values.astype(float)



n = len(dataset)
fig, axs = plt.subplots(n, 2, sharey=False, figsize=(10, 4 * n))
# fig, axs = plt.subplots(n, 2, sharey=True, figsize=(10, 4 * n))
if n == 1:
    axs = np.expand_dims(
        axs, axis=0
    )  # ensure consistent indexing for single row

timepoints = dataset._get_time_points()
gene_labels = dataset.genes
# for i in range(n):
for i, gene in enumerate(gene_labels):
    env, expressions = dataset[i]
    # Plot ground truth
    df_real = pd.DataFrame(
        expressions.detach().numpy(), columns=timepoints, index=gene_labels
    )
    ExpressionData(df_real).plot_expressions(axis=axs[i, 0])
    axs[i, 0].set_title(f"Measured (Sample {i})")

    # Simulate and plot prediction
    # self.model.set_conditions(env.unsqueeze(0))
    # output = self.integrate_model(
    #     initial_states_tensor=expressions[:, 0].unsqueeze(0)
    # )
    # display(env, type(env))
    # output = integrateODE(gene)
    # output = torch.tensor(output.y, dtype=torch.float32)
    # display(type(output), output.shape)
    # df_pred = pd.DataFrame(
    #     output.squeeze(0).detach().numpy(),
    #     columns=timepoints,
    #     index=gene_labels,
    # )
    # ExpressionData(df_pred).plot_expressions(axis=axs[i, 1])
    
    param_dict_mutant = sbml_param_dict.copy()
    for key in param_dict_mutant.keys():
        if key.startswith("a_") and key.endswith(gene):
            param_dict_mutant[key] = 0.0
    solution_mutant = solve_ivp(build_vectorized_system(symbolic_ODE_dict, param_dict_mutant), (0, 1000), y0, t_eval=np.linspace(0, 1000, 200), method="RK45")

    # Plot the results
    for j, node in enumerate(node_names):
        axs[i, 1].plot(solution_mutant.t, solution_mutant.y[j], label=node)
    axs[i, 1].set_title(f"Mutant: {gene}")
    # axes[idx].set_ylabel("Concentration")
    # axes[idx].legend(loc='upper right')
    # axs[i, 1].set_title(f"Model (Sample {i})")
    
    axs[i, 1].set_title(f"Model (Sample {i})")

plt.tight_layout()
    # return fig, axs

In [ ]:
# # Load knockout timeseries data
# knockout_df = pd.read_csv("tests/genenetweaver_outputs/test_scalefree_knockout_timeseries.tsv", sep="\t")
# display(knockout_df)
# # Assume columns: 'Time', 'Node_0', ..., 'Node_9', 'Knockout' (Knockout node name for each experiment)
# knockout_nodes = knockout_df['Knockout'].unique()

# fig, axes = plt.subplots(len(knockout_nodes), 2, figsize=(14, 4 * len(knockout_nodes)), sharex=True)
# if len(knockout_nodes) == 1:
#     axes = np.array([axes])  # Ensure axes is always 2D

# for idx, knockout_node in enumerate(knockout_nodes):
#     # Get timeseries for this knockout
#     df_knock = knockout_df[knockout_df['Knockout'] == knockout_node]
#     duration = df_knock['Time'].values
#     # Plot experimental timeseries
#     for node in node_names:
#         axes[idx, 0].plot(duration, df_knock[node].values, label=node)
#     axes[idx, 0].set_title(f"Experimental: Knockout {knockout_node}")
#     axes[idx, 0].set_ylabel("Concentration")
#     axes[idx, 0].legend(loc='upper right')

#     # Prepare knockout param dict: set all parameters for knockout_node to 0
#     knockout_param_dict = sbml_param_dict.copy()
#     for k in list(knockout_param_dict.keys()):
#         if k.endswith(f"_{knockout_node}"):
#             knockout_param_dict[k] = 0.0

#     # Simulate model with knockout
#     def ode_system_knockout(t, y):
#         dydt = np.zeros(len(node_names))
#         s = np.append(y, 1.0)
#         for i, name in enumerate(node_names):
#             dydt[i] = ODE_lambda_dict[name](s, knockout_param_dict)
#         return dydt

#     y0_knock = df_knock.iloc[0][node_names].values.astype(float)
#     t_span_knock = (duration[0], duration[-1])
#     t_eval_knock = duration
#     sol_knock = solve_ivp(ode_system_knockout, t_span_knock, y0_knock, t_eval=t_eval_knock, method="LSODA")

#     # Plot model output
#     for i, node in enumerate(node_names):
#         axes[idx, 1].plot(sol_knock.t, sol_knock.y[i], label=node)
#     axes[idx, 1].set_title(f"Model: Knockout {knockout_node}")
#     axes[idx, 1].set_ylabel("Concentration")
#     axes[idx, 1].legend(loc='upper right')

# for ax in axes[-1, :]:
#     ax.set_xlabel("Time")
# plt.tight_layout()
# plt.show()

In [ ]:
# import sympy

# # 1) Define parameter dictionary
# param_dict = {
#     "pSPA": 1.0, "dSPA": 0.1, "assCOP1SPA1": 0.2, "dissCOP1SPA1": 0.05,
#     "pCOP1": 1.0, "dCOP1": 0.1, "iCOP1_light": 0.1, "iCOP1_drought": 0.1,
#     "iCOP1_heat_shoot": 1.0, "pCOP1_heat_shoot": 1.0, "iCOP1_SPA1_auto": 0.1,
#     # Add any other missing parameters here
# }

# # 2) Define ODEs as lambdas that take the full state‐dict
# ODE_dict = {
#     "spa1_shoot": lambda s: (
#         param_dict["pSPA"]
#         - param_dict["dSPA"] * s["spa1_shoot"]
#         - param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
#         + param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
#     ),
#     "cop1_shoot": lambda s: (
#         param_dict["pCOP1"] * param_dict["pCOP1_heat_shoot"]**s["heat"]
#         - param_dict["dCOP1"] * s["cop1_shoot"]
#         - (param_dict["iCOP1_light"]*s["light"] + param_dict["iCOP1_drought"]*s["drought"])
#           * param_dict["iCOP1_heat_shoot"]**s["heat"] * s["cop1_shoot"]
#         - param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
#         + param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
#     ),
#     "cop1_spa1_shoot": lambda s: (
#         param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
#         - param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
#         - param_dict["iCOP1_SPA1_auto"] * s["cop1_spa1_shoot"]**2
#     ),
#     # … define hy5_shoot, hy5p_shoot, spa1_root, … etc. …
# }

# input_dict = {
#     "light": lambda t: (t % 24 <= 16),  # light on for 16 hours
#     "heat": lambda t: 0,                 # no heat stress
#     "drought": lambda t: 0                # no drought stress
# }

# def combine_ODE_dict_and_param_dict(ODE_dict, param_dict):
#     """
#     Returns a new ODE_dict where each lambda takes a state dict and uses the correct param subdict.
#     param_dict should be a dict of dicts, with keys matching ODE_dict.
#     """
#     combined_ODE_dict = {}
#     for k, ode_fn in ODE_dict.items():
#         params = param_dict.get(k, {})
#         # Wrap the original lambda to inject the correct param subdict
#         combined_ODE_dict[k] = (lambda ode_fn=ode_fn, params=params: (lambda s: ode_fn(s, params)))()
#     return combined_ODE_dict

# # Example usage:
# # combined_ODE_dict = combine_ODE_dict_and_param_dict(ODE_dict, param_dict)


# # 3) Get a list of the dependencies of each equation
# # dependencies = {
# #     "spa1_shoot":   ["cop1_shoot", "cop1_spa1_shoot"],
# #     "cop1_shoot":   ["heat", "light", "drought", "spa1_shoot", "cop1_spa1_shoot"],
# #     "cop1_spa1_shoot": ["cop1_shoot", "spa1_shoot"],
# # }

# def build_network_from_ODE_dict(ODE_dict: dict, input_dict: dict, remove_linear_self_loops: bool = True):
#     dependencies = {}
#     gene_names = list(ODE_dict.keys())
#     input_names = list(input_dict.keys()) if input_dict is not None else ["light", "heat", "drought"]
#     for var, ODE in ODE_dict.items():
#         symbolic_vars = sympy.symbols(gene_names + input_names)
#         # Build a dict for the lambda
#         state_dict = {v: symbolic_vars[i] for i, v in enumerate(gene_names)}
#         state_dict.update({name: symbolic_vars[len(gene_names) + j] for j, name in enumerate(input_names)})
#         symbolic_expr = sympy.sympify(ODE(state_dict))
#         # Find all dependencies
#         deps = [
#             str(sym) for sym in symbolic_expr.free_symbols
#             if str(sym) in gene_names or str(sym) in input_names
#         ]
#         # Only add self-loop if dependence on self is higher order (nonlinear)
#         if remove_linear_self_loops:
#             if var in deps:
#                 var_sym = symbolic_vars[gene_names.index(var)]
#                 # Check if the highest power of var is > 1
#                 if sympy.degree(symbolic_expr, gen=var_sym) <= 1:
#                     deps.remove(var)
#         dependencies[var] = deps

#     G = nx.DiGraph()
#     for var, ODE in ODE_dict.items():
#         G.add_node(var, ODE=ODE, type="network")
    
#     for input_node, time_series_func in input_dict.items():
#         G.add_node(input_node, time_series_func=time_series_func, type="input")
    
#     for var, deps in dependencies.items():
#         for d in deps:
#             G.add_edge(d, var)
#     return G

# G = build_network_from_ODE_dict(ODE_dict, input_dict)



# # 5) Write a wrapper to produce the derivative‐vector for scipy
# def network_ODE(t, y):
#     state = dict(zip(ODE_dict.keys(), y))
#     # Update state with input values at time t
#     for input_name, input_func in input_dict.items():
#         state[input_name] = input_func(t)
#     # Compute derivatives for each ODE variable
#     dydt = np.zeros(len(ODE_dict))
#     for i, (var, ode_func) in enumerate(ODE_dict.items()):
#         dydt[i] = ode_func(state)
#     return dydt


# # 6) Integrate the ODEs
# gene_names = list(ODE_dict.keys())
# y0 = np.zeros(len(gene_names)) # initial conditions
# ODEsolution = solve_ivp(network_ODE, [0, 7*24], y0, method="LSODA")


# # create mechanistic model
# # MM = ComplexMechanisticModel(RegulatoryNetwork(G), param_dict)
# param_dict_torch = {k: torch.nn.Parameter(torch.tensor(v, dtype=torch.float32)) for k, v in param_dict.items()}
# MM = ComplexODEModel(ODE_dict, input_dict, param_dict_torch)
# gene_names = list(ODE_dict.keys())

# # integrate the ODEs
# t_span = (0, 7*24)  # time span for integration
# t_points = torch.linspace(t_span[0], t_span[1], 1000)  # time points for integration
# # Ensure time points are properly spaced to avoid numerical issues
# # Ensure initial conditions are non-negative and consistent with the equations
# y0 = torch.zeros(len(gene_names))  # initial conditions (non-zero positive values)
# # Add error handling to catch potential numerical issues during integration
# # try:
# #     solution = MM.integrate(y0, t_points)
# # except Exception as e:
# #     print(f"Integration failed: {e}")
# #     solution = None
# # print(MM.regulatory_network.nodes, G.nodes)
# MMsolution = MM.integrate(y0, t_points)
# # solution = solve_ivp(MM.forward, t_span=[0, 7*24], y0=y0, method="LSODA")